# Third-Order SK-EFT: Parity Alternation and the Bogoliubov Connection

**Authors:** John Roehm  
**Date:** March 2026  
**Phase:** 3 of the SK-EFT Hawking Paper series

This computational companion extends the dissipative SK-EFT to third order in the derivative expansion, revealing a structural parity alternation theorem and connecting the EFT to the microscopic Bogoliubov dispersion. All formulas match the Lean 4 formalization (72 total theorems, zero sorry).

## Setup: Imports and Experimental Parameters

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import sys, os

# Add project root for imports
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

from src.core.constants import (
    HBAR,
    K_B,
    ATOMS,
    EXPERIMENTS,
    COLORS as _EXP_COLORS,
    ARISTOTLE_THEOREMS,
    TOTAL_THEOREMS,
    get_bec_parameters,
    get_all_experiments,
)
from src.core.visualizations import (
    COLORS,
    LAYOUT_TEMPLATE,
    apply_layout,
)  # Full palette (experiments + physics categories)
from src.core.formulas import (
    count_coefficients,
    enumerate_monomials,
    damping_rate,
    dispersive_correction,
    first_order_correction,
    second_order_correction,
    third_order_correction,
    beliaev_damping_rate,
    beliaev_transport_coefficients,
)
from src.core.transonic_background import solve_transonic_background
from src.second_order.enumeration import (
    analyze_order,
    parity_alternation,
    third_order_analysis,
)
from src.second_order.coefficients import (
    FirstOrderCoeffs,
    SecondOrderCoeffs,
    ThirdOrderCoeffs,
    FullCoeffsThrough3,
    hawking_correction_third_order,
)

# Build experiments dict from single source of truth
experiments = {}
for name in EXPERIMENTS:
    params = get_bec_parameters(name)
    bg = solve_transonic_background(params)

    p = {}
    p["atom"] = ATOMS[EXPERIMENTS[name]["atom"]]["label"]
    p["mass"] = params.mass
    p["a_s"] = params.scattering_length
    p["n_upstream"] = params.density_upstream
    p["v_upstream"] = params.velocity_upstream
    p["omega_perp"] = params.omega_perp

    # Derived from solver
    p["c_s"] = bg.sound_speed[0]
    p["xi"] = HBAR / (params.mass * p["c_s"])
    p["kappa"] = bg.surface_gravity
    p["D"] = bg.adiabaticity
    p["T_H"] = bg.hawking_temp
    p["color"] = _EXP_COLORS[name]

    # First-order transport coefficients
    transport = beliaev_transport_coefficients(
        p["n_upstream"], p["a_s"], p["kappa"], p["c_s"], p["xi"]
    )
    p["gamma_1"] = transport["gamma_1"]
    p["gamma_2"] = transport["gamma_2"]
    p["gamma_2_1"] = transport["gamma_2_1"]
    p["gamma_2_2"] = transport["gamma_2_2"]
    p["Gamma_Bel"] = transport["Gamma_Bel"]
    p["k_H"] = transport["k_H"]

    # Third-order transport coefficients (dimensional estimate)
    # Extra derivative order -> extra factor of xi/c_s
    suppression_3 = (p["xi"] / p["c_s"]) ** 2
    p["gamma_3_1"] = p["Gamma_Bel"] * suppression_3 / (2 * p["k_H"] ** 4)
    p["gamma_3_2"] = p["gamma_3_1"]  # order-of-magnitude estimate
    p["gamma_3_3"] = p["gamma_3_1"]  # order-of-magnitude estimate

    # Corrections
    p["delta_disp"] = dispersive_correction(p["D"])
    p["delta_diss"] = first_order_correction(p["Gamma_Bel"], p["kappa"])

    experiments[name] = p

# Print summary
print("Phase 3 Technical: Parameters from src.core (single source of truth)")
print("=" * 80)
for name, p in experiments.items():
    print(f"\n{name} ({p['atom']})")
    print(f"  c_s = {p['c_s'] * 1e3:.3f} mm/s, xi = {p['xi'] * 1e6:.3f} um")
    print(f"  kappa = {p['kappa']:.1f} s^-1, D = {p['D']:.4f}")
    print(f"  T_H = {p['T_H'] * 1e9:.3f} nK")
    print(f"  gamma_1 = {p['gamma_1']:.4e} m^2/s")
    print(f"  gamma_2_1 = {p['gamma_2_1']:.4e} m^3/s")
    print(f"  gamma_3_1 = {p['gamma_3_1']:.4e} m^4/s")
    print(f"  delta_disp = {p['delta_disp']:.4e}, delta_diss = {p['delta_diss']:.4e}")

## Section 1: Third-Order Monomial Enumeration

At EFT order $N=3$, the derivative level is $L = N+1 = 4$. We enumerate all candidate monomials $\psi_a \cdot \partial_t^m \partial_x^n \psi_r$ with $m+n=4$, then apply the T-reversal constraint ($m$ even) to determine the surviving transport coefficients.

In [ ]:
# Third-order monomial enumeration
r3 = analyze_order(3, verbose=False)

print("Third-Order Monomial Enumeration (N=3, L=4)")
print("=" * 70)
print(f"{'Monomial':<30} {'(m, n)':<10} {'m even?':<10} {'n even?':<10} {'Status':<15}")
print("-" * 70)

for d in r3.all_real_monomials:
    d_str = repr(d)
    monomial_str = f"psi_a . {d_str + ' ' if d_str else ''}psi_r"
    m_ok = "Yes" if d.t_parity_even else "No"
    n_ok = "Yes" if d.x_parity_even else "No"
    status = "SURVIVES" if d.t_parity_even else "KILLED (T-rev)"
    print(
        f"{monomial_str:<30} ({d.n_t}, {d.n_x}){'':<4} {m_ok:<10} {n_ok:<10} {status:<15}"
    )

print(f"\n5 candidates -> T-reversal kills 2 -> {r3.n_transport_no_parity} survive")
print(
    f"Surviving monomials: (0,4), (2,2), (4,0) -- ALL have even n (parity-preserving)"
)

## Section 2: The Parity Alternation Theorem

**The key structural result of Phase 3.** At each EFT order $N$:
- **Odd $N$** (1, 3, 5, ...): derivative level $L = N+1$ is even, so $n = L - m$ is even when $m$ is even. *All* surviving monomials are parity-preserving $\Rightarrow$ corrections exist universally.
- **Even $N$** (2, 4, 6, ...): $L$ is odd, so $n$ is odd when $m$ is even. *All* surviving monomials require broken spatial parity $\Rightarrow$ corrections are flow-only.

Lean: `parity_preserving_at_odd_order`, `parity_breaking_at_even_order` (ThirdOrderSK.lean)

In [ ]:
# Parity alternation table: N = 1..8
print("Parity Alternation Theorem")
print("=" * 90)
print(
    f"{'Order N':<10} {'Level L':<10} {'Count':>8} {'Count':>10} {'Parity':>18} {'Type':<20}"
)
print(f"{'':10} {'':10} {'(no par.)':>8} {'(with par.)':>10} {'':>18} {'':20}")
print("-" * 90)

for N in range(1, 9):
    pa = parity_alternation(N)
    parity_type = "UNIVERSAL" if not pa["requires_parity_breaking"] else "FLOW-ONLY"
    n_type = (
        "even n (preserving)"
        if not pa["requires_parity_breaking"]
        else "odd n (breaking)"
    )
    print(
        f"{N:<10} {pa['L']:<10} {pa['count_no_parity']:>8} {pa['count_with_parity']:>10} {n_type:>18}   {parity_type:<20}"
    )

print("\nPattern: UNIVERSAL -> FLOW-ONLY -> UNIVERSAL -> FLOW-ONLY -> ...")
print("Lean: parity_preserving_at_odd_order, parity_breaking_at_even_order")

In [ ]:
# viz-ref: fig_parity_alternation_heatmap
# Visual: heatmap of monomial structure at each order

orders = list(range(1, 9))
max_L = max(N + 1 for N in orders)  # = 9

# Build a matrix: rows = orders N, columns = m values (0..max_L)
# Cell value: 0 = not present, 1 = survives (universal), 2 = survives (flow-only), -1 = killed
matrix = []
m_labels = [str(m) for m in range(max_L + 1)]
n_labels = [f"N={N}" for N in orders]

z_data = []
hover_data = []
for N in orders:
    L = N + 1
    pa = parity_alternation(N)
    row = []
    hover_row = []
    for m in range(max_L + 1):
        n = L - m
        if n < 0 or m > L:
            row.append(np.nan)  # outside this order
            hover_row.append("")
        elif m % 2 != 0:
            row.append(-1)  # killed by T-reversal
            hover_row.append(f"N={N}: (m={m},n={n}) KILLED by T-reversal")
        else:
            if n % 2 == 0:
                row.append(1)  # universal
                hover_row.append(f"N={N}: (m={m},n={n}) UNIVERSAL (even n)")
            else:
                row.append(2)  # flow-only
                hover_row.append(f"N={N}: (m={m},n={n}) FLOW-ONLY (odd n)")
    z_data.append(row)
    hover_data.append(hover_row)

# Custom colorscale: nan=white, -1=grey (killed), 1=green (universal), 2=red (flow-only)
killed_color = COLORS["sensitivity"]  # grey for killed monomials
fig = go.Figure(
    data=go.Heatmap(
        z=z_data,
        x=[f"m={m}" for m in range(max_L + 1)],
        y=[f"N={N}" for N in orders],
        customdata=hover_data,
        hovertemplate="%{customdata}<extra></extra>",
        colorscale=[
            [0.0, killed_color],
            [0.33, killed_color],
            [0.33, COLORS["dispersive"]],  # 1 -> green (universal)
            [0.66, COLORS["dispersive"]],
            [0.66, COLORS["dissipative"]],  # 2 -> red (flow-only)
            [1.0, COLORS["dissipative"]],
        ],
        zmin=-1,
        zmax=2,
        showscale=False,
    )
)

# Add text annotations on each cell
for i, N in enumerate(orders):
    L = N + 1
    for m in range(max_L + 1):
        n = L - m
        if n < 0 or m > L:
            continue
        if m % 2 != 0:
            symbol = "X"
            color = "white"
        elif n % 2 == 0:
            symbol = "U"
            color = "white"
        else:
            symbol = "F"
            color = "white"
        fig.add_annotation(
            x=f"m={m}",
            y=f"N={N}",
            text=symbol,
            showarrow=False,
            font=dict(size=14, color=color, family="monospace"),
        )

apply_layout(
    fig,
    title="Parity Alternation: Monomial Structure at Each EFT Order",
    height=450,
    width=700,
    xaxis_title="Time derivative count m",
    yaxis_title="EFT Order N",
)

# Add legend annotation
fig.add_annotation(
    x=1.0,
    y=1.15,
    xref="paper",
    yref="paper",
    text="U = Universal | F = Flow-only | X = Killed by T-reversal",
    showarrow=False,
    font=dict(size=11),
)

fig.show()

## Section 3: Third-Order Transport Coefficients

The 3 new third-order transport coefficients and their physical interpretations:

| Coefficient | Monomial | Physical interpretation | Units |
|:---:|:---:|:---|:---:|
| $\gamma_{3,1}$ | $\psi_a \cdot \partial_x^4 \psi_r$ | Quartic spatial dispersion (Bogoliubov $k^4$) | $[\text{m}^4/\text{s}]$ |
| $\gamma_{3,2}$ | $\psi_a \cdot \partial_t^2 \partial_x^2 \psi_r$ | Mixed $\omega^2 k^2$ cross-term | $[\text{m}^4/\text{s}]$ |
| $\gamma_{3,3}$ | $\psi_a \cdot \partial_t^4 \psi_r$ | Quartic temporal damping ($\omega^4$) | $[\text{m}^4/\text{s}]$ |

All three are parity-preserving (even $n$) and exist universally, even without background flow.

Lean: `thirdOrder_uniqueness`, `thirdOrder_explicit_monomials` (ThirdOrderSK.lean)

In [ ]:
# Third-order analysis with monomial details
r3_detail = third_order_analysis(verbose=True)

print("\n" + "=" * 70)
print("Third-order coefficient estimates from Beliaev matching:")
print("=" * 70)

for name, p in experiments.items():
    print(f"\n{name} ({p['atom']}):")
    print(
        f"  gamma_3,1 = {p['gamma_3_1']:.4e} m^4/s  (quartic spatial, Bogoliubov k^4)"
    )
    print(f"  gamma_3,2 = {p['gamma_3_2']:.4e} m^4/s  (mixed omega^2 k^2)")
    print(f"  gamma_3,3 = {p['gamma_3_3']:.4e} m^4/s  (quartic temporal omega^4)")
    print(f"  EFT expansion parameter xi/lambda_H = {p['D']:.4f}")

## Section 4: Extended Damping Rate Through Third Order

The full damping rate with all 7 transport coefficients:

$$\Gamma(k) = \underbrace{\gamma_1 k^2 + \gamma_2 \omega^2/c_s^2}_{\text{Order 1}} + \underbrace{\gamma_{2,1} k^3 + \gamma_{2,2} \omega^2 k/c_s^2}_{\text{Order 2}} + \underbrace{\gamma_{3,1} k^4 + \gamma_{3,2} \omega^2 k^2/c_s^2 + \gamma_{3,3} \omega^4/c_s^4}_{\text{Order 3}}$$

In [ ]:
# viz-ref: fig_damping_rate_third_order
# Plot damping rate Gamma(k) with all 7 coefficients through third order

fig = go.Figure()

# Use Steinhauer parameters as reference
p = experiments["Steinhauer"]
k_arr = np.linspace(0.01 * p["k_H"], 5 * p["k_H"], 300)
omega_arr = p["c_s"] * k_arr  # on-shell: omega = c_s * k

# First-order only
Gamma_1 = damping_rate(k_arr, omega_arr, p["c_s"], p["gamma_1"], p["gamma_2"])

# Through second order
Gamma_12 = damping_rate(
    k_arr,
    omega_arr,
    p["c_s"],
    p["gamma_1"],
    p["gamma_2"],
    p["gamma_2_1"],
    p["gamma_2_2"],
)

# Through third order
Gamma_123 = damping_rate(
    k_arr,
    omega_arr,
    p["c_s"],
    p["gamma_1"],
    p["gamma_2"],
    p["gamma_2_1"],
    p["gamma_2_2"],
    p["gamma_3_1"],
    p["gamma_3_2"],
    p["gamma_3_3"],
)

fig.add_trace(
    go.Scatter(
        x=k_arr / p["k_H"],
        y=Gamma_1 / p["kappa"],
        mode="lines",
        name="Order 1 only (k^2)",
        line=dict(color=COLORS["dispersive"], width=2, dash="dash"),
    )
)

fig.add_trace(
    go.Scatter(
        x=k_arr / p["k_H"],
        y=Gamma_12 / p["kappa"],
        mode="lines",
        name="Through order 2 (+k^3)",
        line=dict(color=COLORS["cross"], width=2, dash="dot"),
    )
)

fig.add_trace(
    go.Scatter(
        x=k_arr / p["k_H"],
        y=Gamma_123 / p["kappa"],
        mode="lines",
        name="Through order 3 (+k^4)",
        line=dict(color=COLORS["dissipative"], width=3),
    )
)

# Mark horizon wavenumber
fig.add_vline(
    x=1.0,
    line=dict(color=COLORS["horizon"], width=1.5, dash="dash"),
    annotation_text="k = k_H",
    annotation_position="top right",
)

apply_layout(
    fig,
    title="Damping Rate: All 7 Coefficients Through Third Order (Steinhauer)",
    xaxis_title="k / k_H",
    yaxis_title="Gamma(k) / kappa",
    height=450,
    width=700,
    yaxis_type="log",
)

fig.show()

print(f"At k = k_H: Gamma_1/kappa = {Gamma_1[len(k_arr) // 5] / p['kappa']:.4e}")
print(f"The k^4 contribution becomes significant for k >> k_H")

## Section 5: Third-Order Spectral Correction

The third-order correction $\delta^{(3)}(\omega) \propto \omega^4$ is **even** in frequency, in contrast with $\delta^{(2)}(\omega) \propto \omega^3$ which is **odd**. This spectral parity alternation is a direct consequence of the monomial parity alternation.

Lean: `thirdOrder_spectral_even`, `secondOrder_spectral_odd`, `spectral_parity_alternation` (ThirdOrderSK.lean)

In [ ]:
# viz-ref: fig_spectral_correction_comparison
# Plot delta^(3)(omega) vs delta^(2)(omega) showing spectral parity alternation

p = experiments["Steinhauer"]
omega_arr = np.linspace(-3 * p["kappa"], 3 * p["kappa"], 500)
k_arr_full = np.abs(omega_arr) / p["c_s"]

# Second-order correction: proportional to omega^3 (odd)
delta_2 = second_order_correction(
    k_arr_full, omega_arr, p["c_s"], p["gamma_2_1"], p["gamma_2_2"], p["kappa"]
)

# Third-order correction: proportional to omega^4 (even)
delta_3 = third_order_correction(
    k_arr_full,
    omega_arr,
    p["c_s"],
    p["gamma_3_1"],
    p["gamma_3_2"],
    p["gamma_3_3"],
    p["kappa"],
)

# Normalize for visual comparison
delta_2_max = np.max(np.abs(delta_2[delta_2 != 0])) if np.any(delta_2 != 0) else 1.0
delta_3_max = np.max(np.abs(delta_3[delta_3 != 0])) if np.any(delta_3 != 0) else 1.0

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=(
        "Second-order: odd in omega (omega^3)",
        "Third-order: even in omega (omega^4)",
    ),
)

fig.add_trace(
    go.Scatter(
        x=omega_arr / p["kappa"],
        y=delta_2 / delta_2_max if delta_2_max > 0 else delta_2,
        mode="lines",
        line=dict(color=COLORS["dissipative"], width=2.5),
        name="delta^(2)(omega) ~ omega^3",
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=omega_arr / p["kappa"],
        y=delta_3 / delta_3_max if delta_3_max > 0 else delta_3,
        mode="lines",
        line=dict(color=COLORS["dispersive"], width=2.5),
        name="delta^(3)(omega) ~ omega^4",
    ),
    row=1,
    col=2,
)

# Add zero line
for col in [1, 2]:
    fig.add_hline(
        y=0, line=dict(color=COLORS["horizon"], width=0.8, dash="dot"), row=1, col=col
    )

fig.update_xaxes(title_text="omega / kappa", row=1, col=1)
fig.update_xaxes(title_text="omega / kappa", row=1, col=2)
fig.update_yaxes(title_text="delta (normalized)", row=1, col=1)
fig.update_yaxes(title_text="delta (normalized)", row=1, col=2)

apply_layout(
    fig,
    title="Spectral Parity Alternation: Odd N -> Even omega-Power, Even N -> Odd omega-Power",
    height=400,
    width=900,
    showlegend=False,
)

fig.show()

print("Lean: spectral_parity_alternation -- proved for all N")
print("  delta^(N)(omega) has parity (-1)^N in omega")

## Section 6: Kappa-Crossing Analysis

The dispersive and dissipative corrections scale differently with $\kappa$:
- $|\delta_{\text{disp}}| = (\pi/6)(\kappa \xi / c_s)^2 \propto \kappa^2$
- $\delta_{\text{diss}} = A \cdot \kappa \propto \kappa^1$

There exists a unique crossing point $\kappa^* = 6A c_s^2 / (\pi \xi^2)$ where $|\delta_{\text{disp}}| = \delta_{\text{diss}}$.

Lean: `kappaCrossing_pos`, `kappa_crossing_is_crossing` (HawkingUniversality.lean)

In [ ]:
# viz-ref: fig_kappa_crossing
# Plot |delta_disp| and delta_diss vs kappa, show crossing point

fig = go.Figure()

for name, p in experiments.items():
    kappa_range = np.logspace(-1, 5, 500)  # s^-1

    # Dispersive correction: |delta_disp| = (pi/6) * (kappa * xi / c_s)^2
    D_arr = kappa_range * p["xi"] / p["c_s"]
    abs_delta_disp = (np.pi / 6) * D_arr**2

    # Dissipative correction: delta_diss = A * kappa
    # where A = Gamma_Bel / kappa^2 * kappa / kappa = Gamma_Bel / kappa
    # More precisely: delta_diss = Gamma(k_H)/kappa, where Gamma scales as kappa^2
    # So A = Gamma_Bel / (p["kappa"]^2) * kappa (since Gamma ~ kappa^2 from k_H^2)
    # Simplification: delta_diss(kappa) = (gamma_1 + gamma_2) * kappa / c_s^2
    A_coeff = (p["gamma_1"] + p["gamma_2"]) / p["c_s"] ** 2
    delta_diss_arr = A_coeff * kappa_range

    # Crossing point
    kappa_star = 6 * A_coeff * p["c_s"] ** 2 / (np.pi * p["xi"] ** 2)

    fig.add_trace(
        go.Scatter(
            x=kappa_range,
            y=abs_delta_disp,
            mode="lines",
            line=dict(color=p["color"], width=2, dash="dash"),
            name=f"|delta_disp| ({name})",
            legendgroup=name,
        )
    )

    fig.add_trace(
        go.Scatter(
            x=kappa_range,
            y=delta_diss_arr,
            mode="lines",
            line=dict(color=p["color"], width=2),
            name=f"delta_diss ({name})",
            legendgroup=name,
        )
    )

    # Mark crossing point
    delta_star = A_coeff * kappa_star
    fig.add_trace(
        go.Scatter(
            x=[kappa_star],
            y=[delta_star],
            mode="markers",
            marker=dict(
                size=10,
                color=p["color"],
                symbol="diamond",
                line=dict(width=2, color="black"),
            ),
            name=f"kappa* = {kappa_star:.1f} ({name})",
            legendgroup=name,
        )
    )

apply_layout(
    fig,
    title="kappa-Crossing: Dispersive vs. Dissipative Dominance",
    xaxis_title="Surface gravity kappa [s^-1]",
    yaxis_title="|Correction|",
    xaxis_type="log",
    yaxis_type="log",
    height=500,
    width=800,
)

fig.show()

print("\nkappa-crossing values:")
for name, p in experiments.items():
    A_coeff = (p["gamma_1"] + p["gamma_2"]) / p["c_s"] ** 2
    kappa_star = 6 * A_coeff * p["c_s"] ** 2 / (np.pi * p["xi"] ** 2)
    print(f"  {name}: kappa* = {kappa_star:.2e} s^-1")
    print(f"    Below kappa*: dissipative dominates; above: dispersive dominates")
print("\nLean: kappa_crossing_is_crossing -- proved crossing equality")

## Section 7: Spin-Sonic Enhancement

In a two-component BEC (Trento configuration), spin waves propagate at $c_{\text{spin}} \ll c_{\text{density}}$. The dissipative correction is enhanced by a factor $(c_{\text{density}}/c_{\text{spin}})^2$, potentially achieving $O(100)$ amplification.

Lean: `spinSonicEnhancement_pos`, `spinSonic_enhancement_exact` (HawkingUniversality.lean)

In [ ]:
# viz-ref: fig_spin_sonic_enhancement
# Plot enhancement factor (c_d/c_s)^2 vs velocity ratio

c_ratio = np.logspace(0, 2.5, 300)  # c_density / c_spin = 1 to ~300
enhancement = c_ratio**2

# Baseline delta_diss from Steinhauer
delta_diss_baseline = experiments["Steinhauer"]["delta_diss"]
delta_diss_enhanced = delta_diss_baseline * enhancement

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=c_ratio,
        y=delta_diss_enhanced,
        mode="lines",
        line=dict(color=COLORS["dissipative"], width=3),
        name="delta_diss x (c_dens/c_spin)^2",
        fill="tozeroy",
        fillcolor="rgba(230, 57, 70, 0.1)",
    )
)

# Mark experiments
markers = [
    (1, delta_diss_baseline, "Single-component\n(Steinhauer)", COLORS["Steinhauer"]),
    (
        10,
        delta_diss_baseline * 100,
        "Trento projected\n(c ratio ~ 10)",
        COLORS["Trento"],
    ),
    (
        30,
        delta_diss_baseline * 900,
        "Optimistic\n(c ratio ~ 30)",
        COLORS["dissipative"],
    ),
]
for x, y, label, color in markers:
    fig.add_trace(
        go.Scatter(
            x=[x],
            y=[y],
            mode="markers+text",
            marker=dict(size=12, color=color, line=dict(width=2, color="black")),
            text=[label],
            textposition="top center",
            textfont=dict(size=11),
            showlegend=False,
        )
    )

# Sensitivity thresholds
for val, label in [
    (0.1, "10% sensitivity"),
    (0.01, "1% (near-term)"),
    (0.001, "0.1% (projected)"),
]:
    fig.add_hline(
        y=val,
        line=dict(color=COLORS["sensitivity"], width=1.5, dash="dash"),
        annotation_text=label,
        annotation_position="right",
        annotation_font=dict(size=10, color="#888"),
    )

apply_layout(
    fig,
    title="Spin-Sonic Enhancement of Dissipative Corrections",
    xaxis_title="Velocity ratio c_density / c_spin",
    yaxis_title="|delta_diss|",
    xaxis_type="log",
    yaxis_type="log",
    height=450,
    width=700,
    yaxis_range=[-7, 0],
)

fig.show()

print(f"Baseline delta_diss (Steinhauer) = {delta_diss_baseline:.4e}")
print(
    f"At c_ratio = 10: enhancement = 100x -> delta_diss = {delta_diss_baseline * 100:.4e}"
)
print(
    f"At c_ratio = 30: enhancement = 900x -> delta_diss = {delta_diss_baseline * 900:.4e}"
)

## Section 8: Bogoliubov Coefficient Bound

The perturbative bound on the dissipative correction: $\delta_{\text{diss}} \leq 1$ ensures the Bogoliubov coefficient modification is controlled.

When $\Gamma_H \leq \kappa$ (damping rate bounded by surface gravity), the first-order correction satisfies $\delta_{\text{diss}} = \Gamma_H / \kappa \leq 1$.

Lean: `bogoliubov_correction_bounded`, `bogoliubov_correction_perturbative` (WKBAnalysis.lean)

In [ ]:
# Bogoliubov coefficient bound check
print("Perturbative Bound: delta_diss = Gamma_H / kappa <= 1")
print("=" * 60)
print(
    f"{'Experiment':<15} {'Gamma_H [s^-1]':>15} {'kappa [s^-1]':>15} {'delta_diss':>12} {'Bound':>8}"
)
print("-" * 60)

for name, p in experiments.items():
    bound_ok = "OK" if p["delta_diss"] <= 1 else "VIOLATED"
    print(
        f"{name:<15} {p['Gamma_Bel']:>15.4e} {p['kappa']:>15.1f} {p['delta_diss']:>12.4e} {bound_ok:>8}"
    )

print(f"\nAll experiments satisfy delta_diss << 1 (deeply perturbative).")
print(f"EFT is well-controlled: corrections are small perturbations.")
print(f"\nLean: bogoliubov_correction_perturbative")
print(f"  Proves: Gamma_H <= kappa -> delta_diss <= 1")

## Section 9: Cumulative Coefficient Count and Lean Summary

Through three orders, the SK-EFT has 7 independent transport coefficients. The Lean formalization for Phase 3 adds 14 new theorems in `ThirdOrderSK.lean` plus 7 enhancements in `WKBAnalysis.lean` and `HawkingUniversality.lean`.

In [ ]:
# Cumulative coefficient count table
print("Cumulative Transport Coefficient Count")
print("=" * 80)
print(
    f"{'Order N':<10} {'New':>6} {'Cumulative':>12} {'Parity-preserving':>20} {'Flow-only':>12}"
)
print("-" * 80)

cumul = 0
cumul_pp = 0
cumul_fo = 0
for N in range(1, 9):
    pa = parity_alternation(N)
    new = pa["count_no_parity"]
    cumul += new
    if not pa["requires_parity_breaking"]:
        cumul_pp += new
        pp_new = new
        fo_new = 0
    else:
        cumul_fo += new
        pp_new = 0
        fo_new = new
    marker = " <-- Phase 3" if N == 3 else ""
    print(f"{N:<10} {new:>6} {cumul:>12} {cumul_pp:>20} {cumul_fo:>12}{marker}")

print(f"\nLean: cumulative_count_through_3 proves 2 + 2 + 3 = 7")
print(f"Lean: cumulative_parity_preserving_through_3 proves 2 + 0 + 3 = 5")

In [ ]:
# Lean theorem summary for Phase 3
print("Phase 3 Lean Theorem Summary")
print("=" * 70)

thirdorder_theorems = [
    "thirdOrder_uniqueness",
    "parity_preserving_at_odd_order",
    "parity_breaking_at_even_order",
    "thirdOrder_parity_count",
    "secondOrder_parity_count_zero",
    "fourthOrder_parity_count_zero",
    "fifthOrder_parity_count",
    "thirdOrder_spatial_derivative_counts",
    "thirdOrder_explicit_monomials",
    "thirdOrder_spectral_even",
    "secondOrder_spectral_odd",
    "spectral_parity_alternation",
    "cumulative_parity_preserving_through_3",
    "parity_fraction_through_3",
]

enhancement_theorems = [
    "kappaCrossing_pos",
    "kappa_crossing_is_crossing",
    "spinSonicEnhancement_pos",
    "spinSonic_enhancement_exact",
    "bogoliubov_correction_bounded",
    "bogoliubov_correction_perturbative",
    "bogoliubov_superluminal",
]

print(f"\nThirdOrderSK.lean: {len(thirdorder_theorems)} theorems")
for t in thirdorder_theorems:
    print(f"  {t}")

print(
    f"\nEnhancement theorems (HawkingUniversality + WKBAnalysis): {len(enhancement_theorems)}"
)
for t in enhancement_theorems:
    print(f"  {t}")

print(f"\nPhase 3 new theorems: {len(thirdorder_theorems) + len(enhancement_theorems)}")
print(f"Previous total (Phases 1+2): {TOTAL_THEOREMS}")
new_total = TOTAL_THEOREMS + len(thirdorder_theorems) + len(enhancement_theorems)
print(
    f"Cumulative total: {TOTAL_THEOREMS} + {len(thirdorder_theorems)} + {len(enhancement_theorems)} = {new_total}"
)
print(f"All proved, zero sorry.")